# 🔍 Práctica de comparación de LLMs · Modelo asignado: `google/gemma-2-2b-it`

**Modelo:** `google/gemma-2-2b-it` (~2,6B parámetros, junio 2024)
**Tarea elegida por el grupo:** Clasificación de sentimiento de reseñas de producto (positiva / negativa / neutra).
**Estrategia:** comparar **3 prompts** × **5 repeticiones** sobre la misma reseña, con `temperature > 0`.

> ⚠️ **Requisitos de hardware:** Usa una GPU T4 o superior en Colab.
> Ir a `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`

> ⚠️ **Acceso al modelo:** `google/gemma-2-2b-it` es un modelo *gated*. Antes de ejecutar:
> 1. Aceptar las condiciones en https://huggingface.co/google/gemma-2-2b-it (botón *Acknowledge license*).
> 2. Generar un token de lectura en https://huggingface.co/settings/tokens.
> 3. En Colab, guardarlo como secret `HF_TOKEN` (icono 🔑 de la barra lateral, activar *Notebook access*).

---

### Identificación del modelo

| Campo | Valor |
|---|---|
| Nombre | `google/gemma-2-2b-it` |
| Familia | Gemma 2 (Google DeepMind) |
| Tamaño | ~2.600 millones de parámetros |
| Variante | *Instruction-tuned* (`-it`) |
| Fecha de publicación | Junio 2024 |
| Tensor type | BF16 |
| Ventana de contexto | 8.192 tokens |
| Licencia | Gemma Terms of Use (gated) |

**Motivo de elección:** equilibrio entre coste computacional y calidad. Cabe en la GPU gratuita de Colab y al estar ajustado por instrucciones sigue prompts estructurados con razonable fidelidad. Sirve como referente "ligero" frente a modelos de 3B–8B parámetros que usen los otros miembros del grupo.


---
## 📦 1. Instalación de dependencias

Gemma 2 requiere `transformers >= 4.42`. Forzamos una versión reciente para evitar problemas de compatibilidad.


In [ ]:
# Instalamos las librerías necesarias
!pip install -q -U transformers accelerate huggingface_hub


---
## 🔧 2. Verificación del entorno


In [ ]:
import torch

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    print("✅ Mac con MPS detectado.")
    DEVICE = "mps"
else:
    print("⚠️  No se detectó GPU. El modelo será MUY lento en CPU.")
    DEVICE = "cpu"


---
## 🤖 3. Carga del modelo Gemma 2 2B IT

Se carga en **BF16** (~5 GB de VRAM en T4). No usamos cuantización 4-bit porque el modelo ya es lo bastante pequeño.

Como `google/gemma-2-2b-it` es un modelo *gated*, primero hay que **autenticarse**. Ejecuta la celda siguiente: aparecerá un campo donde pegar tu token de Hugging Face. El token no se guarda en el notebook.


In [ ]:
# Login interactivo: ejecuta esta celda y pega tu token de Hugging Face cuando aparezca el campo.
# El token NO se guarda en el notebook — queda solo en memoria de la sesión.
from huggingface_hub import login

login()


In [ ]:
# Comprobación rápida de acceso al repo (no descarga los pesos del modelo)
from huggingface_hub import HfApi

info = HfApi().model_info("google/gemma-2-2b-it")
print(f"✅ Acceso concedido al repo. Archivos: {len(info.siblings)}")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_ID = "google/gemma-2-2b-it"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype="auto",   # Gemma 2 → BF16 automáticamente
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("✅ Pipeline listo.")
print(f"   Arquitectura : {pipe.model.config.model_type}")
print(f"   Device       : {pipe.model.device}")
print(f"   Dtype        : {pipe.model.dtype}")
print(f"   Parámetros   : {sum(p.numel() for p in pipe.model.parameters()) / 1e6:.1f} M")


---
## ⚙️ 4. Función de inferencia base

Agrupamos los hiperparámetros en un diccionario `generation_args` y los desempaquetamos al llamar al pipeline. Misma convención que el notebook Phi-3 del grupo.

> ⚠️ **Diferencia importante con el notebook Phi-3:** allí se usa `temperature=0.0, do_sample=False` (modo determinista). En esta práctica el enunciado pide explícitamente `temperature > 0` y repetir varias veces para estudiar variabilidad, así que mantenemos `temperature=0.7`, `top_p=0.9`, `do_sample=True`.

> ⚠️ **Nota sobre el rol `system`:** Phi-3 admite mensajes con `role: "system"`, pero **Gemma 2 no** — su chat template oficial solo acepta `user` y `assistant`. Por eso aquí solo enviamos turnos de `user`.


In [ ]:
def generar_respuesta(prompt: str,
                     temperature: float = 0.7,
                     top_p: float = 0.9,
                     max_new_tokens: int = 400) -> str:
    """
    Genera una respuesta de Gemma 2 2B IT a partir de un prompt de usuario.
    Devuelve solo el texto del asistente (string).
    """
    mensajes = [{"role": "user", "content": prompt}]

    generation_args = {
        "max_new_tokens": max_new_tokens,
        "return_full_text": False,
        "do_sample": True,
        "temperature": temperature,
        "top_p": top_p,
    }

    output = pipe(mensajes, **generation_args)
    return output[0]["generated_text"].strip()


# Prueba rápida
test = generar_respuesta(
    "Saluda en una frase y di que estás listo para clasificar reseñas.",
    max_new_tokens=60,
)
print(test)


---
## 📝 5. Definición del problema y prompts

La reseña es la misma para los tres prompts. Lo único que cambia es la **estrategia de prompt engineering**.


In [ ]:
RESENA = (
    "El producto llegó antes de lo esperado y el embalaje estaba perfecto. "
    "Sin embargo, después de usarlo tres días, dejó de funcionar correctamente. "
    "El servicio de atención al cliente no respondió a mis mensajes. "
    "Muy decepcionante."
)

ETIQUETA_GOLD = "negativa"  # Etiqueta de oro acordada por el grupo

# ---------- Prompt 1: BASE ----------
PROMPT_BASE = f'''Clasifica la siguiente reseña como positiva, negativa o neutra:
"{RESENA}"'''

# ---------- Prompt 2: PLANTILLA CLARA ----------
PROMPT_PLANTILLA = f'''Tarea: Clasifica la reseña de un producto como positiva, negativa o neutra.
Contexto: Eres un sistema de análisis de opiniones para una tienda online.
La clasificación se usará para mejorar el servicio al cliente.
Restricciones:
- Responde únicamente con una de estas etiquetas: positiva, negativa, neutra.
- Añade una justificación de máximo 2 frases.
- No uses información externa a la reseña.
Formato de salida:
Clasificación: [etiqueta]
Justificación: [máximo 2 frases]
Criterio de calidad: La clasificación debe reflejar el sentimiento predominante
del texto, no solo aspectos aislados.
Reseña: "{RESENA}"'''

# ---------- Prompt 3: RAZONAMIENTO Y CONTRASTE ----------
PROMPT_RAZONAMIENTO = f'''Analiza la siguiente reseña de producto paso a paso y determina su clasificación.
Reseña: "{RESENA}"
Sigue estos pasos:
1. Identifica los aspectos positivos mencionados en la reseña.
2. Identifica los aspectos negativos mencionados en la reseña.
3. Considera las tres posibles clasificaciones: positiva, negativa y neutra.
4. Compara el peso de los argumentos a favor de cada una.
5. Escribe una conclusión final justificada con la clasificación elegida.
Formato de respuesta:
Aspectos positivos: ...
Aspectos negativos: ...
Análisis de alternativas: ...
Conclusión y clasificación final: ...'''

PROMPTS = {
    "1_base": PROMPT_BASE,
    "2_plantilla": PROMPT_PLANTILLA,
    "3_razonamiento": PROMPT_RAZONAMIENTO,
}

for nombre, p in PROMPTS.items():
    print(f"--- {nombre} ---")
    print(p[:200] + ("..." if len(p) > 200 else ""))
    print()


---
## 🔬 6. Ejecución del experimento

5 repeticiones por prompt → 15 ejecuciones totales con `temperature=0.7`, `top_p=0.9`.

> Tiempo estimado en T4: ~1–2 minutos.


In [ ]:
import pandas as pd
import time

N_REPETICIONES = 5
TEMPERATURE = 0.7
TOP_P = 0.9

resultados = []
t0 = time.time()

for nombre_prompt, texto_prompt in PROMPTS.items():
    for i in range(1, N_REPETICIONES + 1):
        print(f"[{nombre_prompt}] repetición {i}/{N_REPETICIONES} ...", end=" ", flush=True)
        ti = time.time()
        respuesta = generar_respuesta(
            texto_prompt,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            max_new_tokens=400 if nombre_prompt == "3_razonamiento" else 200,
        )
        dt = time.time() - ti
        print(f"{dt:.1f}s")
        resultados.append({
            "prompt": nombre_prompt,
            "repeticion": i,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "respuesta": respuesta,
            "tiempo_s": round(dt, 2),
        })

df_resultados = pd.DataFrame(resultados)
print(f"\n✅ Total: {len(df_resultados)} ejecuciones en {time.time()-t0:.1f}s")
df_resultados.head()


### Vista rápida de las respuestas


In [ ]:
for nombre in PROMPTS.keys():
    print("=" * 70)
    print(f"PROMPT: {nombre}")
    print("=" * 70)
    print(df_resultados[df_resultados["prompt"] == nombre].iloc[0]["respuesta"])
    print()


---
## 📊 7. Métricas

### 7.1 Métricas objetivas

| Métrica | Cómo se calcula |
|---|---|
| **Etiqueta correcta (1/0)** | Buscamos `positiva` / `negativa` / `neutra` en la respuesta y comparamos con `ETIQUETA_GOLD = "negativa"`. |
| **Cumplimiento del formato (1/0)** | Comprobamos que aparezcan los marcadores que exige cada prompt. |
| **Apartados solicitados (proporción)** | Cuántos de los apartados pedidos aparecen, sobre el total. |
| **Consistencia (% misma etiqueta)** | Sobre las 5 repeticiones del mismo prompt: porcentaje que coincide con la etiqueta más frecuente. |


In [ ]:
import re
import unicodedata

def _normaliza(s: str) -> str:
    s = s.lower()
    s = ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')
    return s

def extraer_etiqueta(respuesta: str) -> str | None:
    """Extrae la etiqueta (positiva / negativa / neutra) de la respuesta del modelo."""
    texto = _normaliza(respuesta)
    patrones = [
        r"clasificacion(?:\s+final)?\s*[:\-]\s*\**\s*(positiva|negativa|neutra)",
        r"conclusion[^\n]*clasificacion[^\n]*[:\-]\s*\**\s*(positiva|negativa|neutra)",
        r"etiqueta\s*[:\-]\s*(positiva|negativa|neutra)",
    ]
    for pat in patrones:
        m = re.search(pat, texto)
        if m:
            return m.group(1)
    m = re.search(r"\b(positiva|negativa|neutra)\b", texto)
    return m.group(1) if m else None


def cumple_formato(respuesta: str, prompt_id: str) -> int:
    t = _normaliza(respuesta)
    if prompt_id == "1_base":
        return int(extraer_etiqueta(respuesta) is not None)
    if prompt_id == "2_plantilla":
        return int(("clasificacion" in t) and ("justificacion" in t))
    if prompt_id == "3_razonamiento":
        return int(
            ("aspectos positivos" in t)
            and ("aspectos negativos" in t)
            and ("alternativas" in t or "analisis de alternativas" in t)
            and ("conclusion" in t)
        )
    return 0


APARTADOS = {
    "1_base": ["etiqueta_clasificacion"],
    "2_plantilla": ["clasificacion", "justificacion"],
    "3_razonamiento": [
        "aspectos positivos",
        "aspectos negativos",
        "analisis de alternativas",
        "conclusion",
    ],
}

def proporcion_apartados(respuesta: str, prompt_id: str) -> float:
    t = _normaliza(respuesta)
    requeridos = APARTADOS[prompt_id]
    if prompt_id == "1_base":
        return 1.0 if extraer_etiqueta(respuesta) else 0.0
    encontrados = sum(1 for r in requeridos if r in t)
    return round(encontrados / len(requeridos), 2)


df_resultados["etiqueta_predicha"] = df_resultados["respuesta"].apply(extraer_etiqueta)
df_resultados["etiqueta_correcta"] = (df_resultados["etiqueta_predicha"] == ETIQUETA_GOLD).astype(int)
df_resultados["formato_ok"] = df_resultados.apply(lambda r: cumple_formato(r["respuesta"], r["prompt"]), axis=1)
df_resultados["apartados"] = df_resultados.apply(lambda r: proporcion_apartados(r["respuesta"], r["prompt"]), axis=1)

df_resultados[["prompt", "repeticion", "etiqueta_predicha", "etiqueta_correcta", "formato_ok", "apartados", "tiempo_s"]]


**Consistencia entre repeticiones** — % de veces que el modelo coincide con la etiqueta más frecuente bajo cada prompt.


In [ ]:
def consistencia(serie):
    if serie.empty:
        return 0.0
    moda = serie.mode().iloc[0] if not serie.mode().empty else None
    if moda is None:
        return 0.0
    return round((serie == moda).mean(), 2)

resumen = (
    df_resultados.groupby("prompt")
      .agg(
          n=("repeticion", "count"),
          acierto_etiqueta=("etiqueta_correcta", "mean"),
          formato_ok=("formato_ok", "mean"),
          apartados_medio=("apartados", "mean"),
          consistencia=("etiqueta_predicha", consistencia),
          etiqueta_moda=("etiqueta_predicha", lambda s: s.mode().iloc[0] if not s.mode().empty else None),
          tiempo_medio_s=("tiempo_s", "mean"),
      )
      .round(2)
)
resumen


### 7.2 Métricas subjetivas (escala 1–5)

Las puntúa el grupo manualmente leyendo las respuestas:

- **Claridad** de la justificación
- **Coherencia** del razonamiento
- **Utilidad** de la respuesta
- **Calidad argumentativa**

Rellenad la plantilla con enteros 1–5. Los `None` se ignoran al promediar.


In [ ]:
evaluacion_subjetiva = []
for _, fila in df_resultados.iterrows():
    evaluacion_subjetiva.append({
        "prompt": fila["prompt"],
        "repeticion": fila["repeticion"],
        "claridad": None,
        "coherencia": None,
        "utilidad": None,
        "calidad_argumentativa": None,
        "comentarios": "",
    })

df_subj = pd.DataFrame(evaluacion_subjetiva)
df_subj


**(Opcional)** Celda de demostración con valores plausibles para ver cómo se calcula la agregación. Sustituidla por puntuaciones reales para la entrega.


In [ ]:
import numpy as np
np.random.seed(42)

mapa_demo = {
    "1_base":          {"claridad": 3, "coherencia": 3, "utilidad": 2, "calidad_argumentativa": 2},
    "2_plantilla":     {"claridad": 4, "coherencia": 4, "utilidad": 4, "calidad_argumentativa": 4},
    "3_razonamiento":  {"claridad": 4, "coherencia": 5, "utilidad": 5, "calidad_argumentativa": 5},
}

for i, row in df_subj.iterrows():
    base = mapa_demo[row["prompt"]]
    for k, v in base.items():
        ruido = np.random.choice([-1, 0, 0, 0, 1])
        df_subj.at[i, k] = int(np.clip(v + ruido, 1, 5))

df_subj


In [ ]:
campos_subj = ["claridad", "coherencia", "utilidad", "calidad_argumentativa"]
resumen_subj = (
    df_subj.groupby("prompt")[campos_subj]
           .mean()
           .round(2)
)
resumen_subj


### 7.3 Tabla comparativa final


In [ ]:
tabla_final = resumen.join(resumen_subj)
tabla_final


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

tabla_final[["acierto_etiqueta", "formato_ok", "apartados_medio", "consistencia"]].plot(
    kind="bar", ax=axes[0], rot=0
)
axes[0].set_title("Métricas objetivas por prompt")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Proporción")
axes[0].legend(loc="lower right", fontsize=8)

tabla_final[["claridad", "coherencia", "utilidad", "calidad_argumentativa"]].plot(
    kind="bar", ax=axes[1], rot=0
)
axes[1].set_title("Métricas subjetivas (1-5) por prompt")
axes[1].set_ylim(0, 5.5)
axes[1].set_ylabel("Puntuación media")
axes[1].legend(loc="lower right", fontsize=8)

plt.tight_layout()
plt.show()


---
## 🔁 8. Análisis de variabilidad

Más allá de la consistencia de la etiqueta, observamos la longitud de las respuestas y su dispersión.


In [ ]:
df_resultados["longitud_chars"] = df_resultados["respuesta"].str.len()
df_resultados["longitud_palabras"] = df_resultados["respuesta"].str.split().str.len()

variabilidad = (
    df_resultados.groupby("prompt")
      .agg(
          long_media_chars=("longitud_chars", "mean"),
          long_std_chars=("longitud_chars", "std"),
          long_media_palabras=("longitud_palabras", "mean"),
          long_std_palabras=("longitud_palabras", "std"),
          etiquetas_distintas=("etiqueta_predicha", lambda s: s.nunique()),
      )
      .round(1)
)
variabilidad


In [ ]:
# Tabla de etiquetas por repetición → de un vistazo se ve qué prompt es más estable
pd.crosstab(df_resultados["prompt"], df_resultados["etiqueta_predicha"], dropna=False)


---
## 💾 9. Exportar resultados a CSV


In [ ]:
import os

output_path_resultados = "resultados_gemma2_2b_it.csv"
output_path_resumen = "resumen_gemma2_2b_it.csv"

df_resultados.to_csv(output_path_resultados, index=False, encoding="utf-8-sig")
tabla_final.to_csv(output_path_resumen, encoding="utf-8-sig")

print(f"✅ Resultados detallados : {os.path.abspath(output_path_resultados)}")
print(f"✅ Resumen por prompt    : {os.path.abspath(output_path_resumen)}")

# Si estamos en Colab, descarga automática; si no, los CSVs quedan en la carpeta del notebook.
try:
    from google.colab import files
    files.download(output_path_resultados)
    files.download(output_path_resumen)
except ImportError:
    print("\n(Ejecutándose en local: los CSVs se han guardado junto al notebook.)")

---
## 🧩 10. Conclusiones (rellenar tras la ejecución)

> Esta sección debe redactarse **después** de ejecutar el notebook, mirando los resultados reales obtenidos en la sesión que entreguéis.

Ideas que conviene incluir:

- **Acierto de etiqueta:** ¿Los tres prompts aciertan la etiqueta `negativa`? ¿En cuál acierta más veces sobre las 5 repeticiones?
- **Formato:** ¿Gemma 2 2B respeta el formato pedido en el prompt 2 (`Clasificación:` / `Justificación:`)? ¿Y los apartados del prompt 3?
- **Estabilidad:** ¿El prompt estructurado reduce la variabilidad respecto al prompt base? Comparar la columna `consistencia` y la `crosstab`.
- **Calidad argumentativa:** ¿El prompt de razonamiento mejora la justificación o solo la alarga?
- **Coste vs beneficio:** ¿Compensa el coste extra (más tokens generados, más tiempo) del prompt 3 frente al 2?
- **Limitaciones del modelo 2B:** Posibles fallos típicos (alucina apartados, mezcla idiomas, no respeta el formato, se "queda en positivo" porque ve el envío rápido al principio…).

---

### 📚 Referencia de prompts y ajustes

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| `do_sample` | `True` | Muestreo estocástico (necesario para variabilidad) |
| `temperature` | `0.7` | Aleatoriedad moderada (requisito del enunciado: > 0) |
| `top_p` | `0.9` | Nucleus sampling |
| `max_new_tokens` | 200–400 | Más para el prompt 3 (razonamiento extendido) |
| `torch_dtype` | `auto` (BF16) | Tipo nativo de Gemma 2 |
| `repeticiones` | 5 por prompt | Para estimar consistencia |

**Comparación con el resto del grupo** (se hará en la presentación común): tabla con un acierto, formato, consistencia y calidad media por (modelo × prompt).
